# M1 arXiv Crawler — Google Colab (CPU-only)

**Trước khi chạy:** Push code lên GitHub của nhóm; kiểm tra 2 mốc tháng 03/09; đọc `README_ARXIV_CRAWLER.md`. Chạy mỗi cell theo thứ tự. Backup vào Google Drive, từng batch 50 bài. Không mở nhiều session chạy song song.


## 1. Clone code từ GitHub
Thay `OWNER` và `REPO` bằng tên GitHub thực tế của nhóm.


In [ ]:
OWNER = 'OWNER'
REPO = 'REPO'
assert OWNER != 'OWNER' and REPO != 'REPO', 'Điền tên OWNER/REPO thật trước'
!git clone https://github.com/{OWNER}/{REPO}.git
%cd {REPO}


## 2. Cài thư viện và kiểm thử offline


In [ ]:
!python -m pip -q install -r src/m1_crawling/requirements-crawler.txt
!python -m pytest -q tests/test_arxiv_crawler.py
!python -m src.m1_crawling.run_arxiv_only --preview


## 3. Mount Google Drive / shared folder
Thư mục bên dưới phải tồn tại trong **MyDrive** của tài khoản chạy Colab và đã share quyền với nhóm.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BACKUP = '/content/drive/MyDrive/arxiv-m1/range3'


## 4. Thử một batch và xác minh ranh giới
Chọn range 3 vì nằm trong một tháng. Kiểm tra toàn bộ giới hạn giao tháng trước khi chạy tất cả.


In [ ]:
!python -m src.m1_crawling.run_arxiv_only --verify-boundaries --range 3 --batch-size 50 --next 1 --backup-dir '{BACKUP}'


## 5. Chạy tiếp 5 batch mỗi lần
Nếu Colab ngắt: mở runtime mới, chạy lại từ đầu đến cell này. Chương trình bỏ qua các batch đã backup. Lưu ý `--range` và thư mục backup phải giữ nguyên.


In [ ]:
!python -m src.m1_crawling.run_arxiv_only --range 3 --batch-size 50 --next 5 --backup-dir '{BACKUP}'


## 6. Xem tiến độ (không tải lại dữ liệu)


In [ ]:
import glob, json, os
files = sorted(glob.glob(BACKUP+'/batch_*.json'))
print('Đã lưu:', len(files), '/ 100 batch (range 3)')
print('Có lỗi:', sum(json.load(open(f))['failed'] for f in files))
print('Dung lượng file nén (GiB):', round(sum(os.path.getsize(f[:-5]+'.tar.gz') for f in files if os.path.exists(f[:-5]+'.tar.gz'))/1024**3, 3))


## 7. Retry bài lỗi sau này, nếu có


In [ ]:
# Bỏ dấu # ở dòng sau nếu cần
# !python -m src.m1_crawling.run_arxiv_only --range 3 --batch-size 50 --next 5 --retry-failed --backup-dir '{BACKUP}'
